# 02 — Legal Document Parsing

This notebook implements the second stage of the **CLARA** document ingestion pipeline.

It transforms the page-level text extracted in `01_pdf_extraction.ipynb` into a structured legal representation centred on articles and document hierarchy.

The parser identifies legal elements such as parts, books, titles, chapters, sections, subsections, article numbers, and article titles. It also removes common extraction artefacts such as recurring page headers and footers and attempts to avoid false article detections originating from tables of contents.

The main output is:

```text
data/processed/<DOCUMENT_ID>/parsed.json
```

This structured representation is used by the chunking stage that follows.

## 1. Imports

The notebook uses standard Python libraries for filesystem access, JSON processing, regular expressions, and frequency analysis, together with Pandas for validation and inspection.

Regular expressions are particularly important in this stage because Portuguese legal documents follow recurring structural conventions that can be detected systematically.

In [1]:
from pathlib import Path
import json
import re
from collections import Counter

import pandas as pd

## 2. Project Paths

The notebook works with the intermediate files generated during PDF extraction.

`PROCESSED_DIR` points to the directory containing one subdirectory per legal document, where the `extracted.json` and subsequent processing artefacts are stored.

In [2]:
PROJECT_ROOT = Path("..").resolve()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print(PROCESSED_DIR)

C:\Users\user\Documents\GitHub\clara-rag-pt\data\processed


## 3. Discovering Extracted Documents

All `extracted.json` files generated by the previous notebook are discovered automatically.

This allows the parser to operate over the complete corpus without requiring each legal document to be configured manually.

In [3]:
extracted_files = sorted(
    PROCESSED_DIR.glob("*/extracted.json")
)

print(f"Foram encontrados {len(extracted_files)} documentos:\n")

for path in extracted_files:
    print(f"- {path.parent.name}")

Foram encontrados 10 documentos:

- CIMI
- CIRC
- CIRS
- CIS
- CIVA
- EBF
- LGT
- OE2026
- RGIT
- RITI


## 4. Legal Structure Detection Patterns

Regular expressions are defined to recognise the main structural elements commonly found in Portuguese legislation.

These patterns cover article headings and hierarchical elements such as parts, books, titles, chapters, sections, and subsections.

The article expression is designed to accommodate common numbering variants, including amended or inserted articles such as `72.º-A`.

In [4]:
ARTICLE_RE = re.compile(
    r"^Artigo\s+"
    r"(?P<number>"
        r"\d+"
        r"(?:\.\d+)?"
        r"\.?"
        r"[ºª]?"
        r"(?:\s*[-–—]\s*[A-Z])?"
    r")"
    r"(?:\s*[—–-]\s+(?P<title>.+))?$",
    re.IGNORECASE
)

PART_RE = re.compile(
    r"^PARTE\s+(.+)$",
    re.IGNORECASE
)

BOOK_RE = re.compile(
    r"^LIVRO\s+(.+)$",
    re.IGNORECASE
)

TITLE_RE = re.compile(
    r"^T[ÍI]TULO\s+(.+)$",
    re.IGNORECASE
)

CHAPTER_RE = re.compile(
    r"^CAP[ÍI]TULO\s+(.+)$",
    re.IGNORECASE
)

SECTION_RE = re.compile(
    r"^SEC[ÇC][ÃA]O\s+(.+)$",
    re.IGNORECASE
)

SUBSECTION_RE = re.compile(
    r"^SUBSEC[ÇC][ÃA]O\s+(.+)$",
    re.IGNORECASE
)

### 4.1 Article Number Normalisation

Legal article identifiers may contain different spacing or dash characters depending on the source PDF.

This function normalises those representations into a consistent format. Consistent identifiers are essential for later metadata generation, chunk identifiers, retrieval evaluation, and source presentation.

In [5]:
def normalize_article_number(value: str) -> str:
    """
    Normaliza o número do artigo.

    Exemplos:
        15.º - A -> 15.º-A
        15.º–A   -> 15.º-A
        72.º     -> 72.º
    """

    value = value.strip()

    value = re.sub(
        r"\s*[-–—]\s*",
        "-",
        value
    )

    return value

## 5. Text Line Normalisation

PDF extraction frequently introduces minor formatting inconsistencies such as non-breaking spaces, soft hyphens, repeated whitespace, or unusual Unicode characters.

`normalize_line()` performs conservative line-level cleaning while preserving the legal content of the document.

In [6]:
def normalize_line(line: str) -> str:
    """
    Normalização mínima de uma linha de texto.
    """

    line = line.replace("\xa0", " ")
    line = line.replace("\u00ad", "")
    
    line = re.sub(r"[ \t]+", " ", line)

    return line.strip()

### 5.1 Converting Pages into Normalised Lines

The extracted text of each page is converted into a list of non-empty, normalised lines.

Working at line level makes it easier to identify legal headings, article boundaries, titles, and recurring page artefacts.

In [7]:
def page_to_lines(text: str) -> list[str]:
    """
    Converte o texto de uma página numa lista
    de linhas normalizadas e não vazias.
    """

    lines = []

    for line in text.splitlines():

        line = normalize_line(line)

        if line:
            lines.append(line)

    return lines

## 6. Detecting Repeated Headers and Footers

Legal PDFs often contain headers, footers, document titles, or other repeated text on many pages.

`detect_repeated_page_lines()` analyses lines near the beginning and end of pages and identifies those that occur frequently enough to be considered page-level artefacts.

Removing these repeated elements reduces noise before article detection.

In [8]:
def detect_repeated_page_lines(
    pages: list[dict],
    edge_lines: int = 3,
    min_ratio: float = 0.20
) -> set[str]:
    """
    Identifica linhas repetidas no início ou fim das páginas.

    Apenas são consideradas as primeiras e últimas
    `edge_lines` linhas de cada página.
    """

    counter = Counter()

    total_pages = len(pages)

    for page in pages:

        lines = page_to_lines(page["text"])

        if not lines:
            continue

        candidates = (
            lines[:edge_lines]
            +
            lines[-edge_lines:]
        )

        # uma linha conta apenas uma vez por página
        for line in set(candidates):
            counter[line] += 1

    minimum_occurrences = max(
        3,
        int(total_pages * min_ratio)
    )

    repeated = {
        line
        for line, count in counter.items()
        if count >= minimum_occurrences
    }

    return repeated

### 6.1 Testing Repeated-Line Detection

The repeated-line detection procedure is first tested on a single extracted document.

This provides a manual inspection point before the same logic is applied automatically to the complete corpus.

In [9]:
test_path = extracted_files[0]

with open(test_path, "r", encoding="utf-8") as f:
    test_document = json.load(f)

repeated_lines = detect_repeated_page_lines(
    test_document["pages"]
)

print(f"Documento: {test_document['document_id']}")
print(f"Linhas repetidas encontradas: {len(repeated_lines)}\n")

for line in sorted(repeated_lines):
    print(repr(line))

Documento: CIMI
Linhas repetidas encontradas: 2

'Código do Imposto Municipal sobre Imóveis (CIMI)'
'Última atualização: Decreto-Lei n.º 49/2025, de 27 de março'


### 6.2 Cleaning Page Lines

`clean_page_lines()` combines line normalisation with the removal of detected recurring headers and footers.

The resulting lines form the cleaned textual input used by the legal parser.

In [10]:
def clean_page_lines(
    text: str,
    repeated_lines: set[str] | None = None
) -> list[str]:

    repeated_lines = repeated_lines or set()

    lines = page_to_lines(text)

    cleaned = []

    for line in lines:

        if line in repeated_lines:
            continue

        cleaned.append(line)

    return cleaned

## 7. Identifying Document Hierarchy

`identify_structure()` checks whether a line represents one of the recognised hierarchical levels of a legal document.

The hierarchy maintained by the parser includes:

```text
Part
└── Book
    └── Title
        └── Chapter
            └── Section
                └── Subsection
```

This metadata is preserved for each article and can later improve traceability and contextual interpretation.

In [11]:
def identify_structure(line: str):

    patterns = [
        ("part", PART_RE),
        ("book", BOOK_RE),
        ("title", TITLE_RE),
        ("chapter", CHAPTER_RE),
        ("section", SECTION_RE),
        ("subsection", SUBSECTION_RE),
    ]

    for structure_type, pattern in patterns:

        match = pattern.match(line)

        if match:
            return {
                "type": structure_type,
                "value": line
            }

    return None

### 7.1 Testing Structural Recognition

A small set of representative legal headings is used to validate the hierarchy and article regular expressions.

This test helps confirm that the parser recognises the expected Portuguese legal-document conventions before processing the corpus.

In [12]:
tests = [
    "PARTE I",
    "LIVRO II",
    "TÍTULO III",
    "CAPÍTULO IV",
    "SECÇÃO II",
    "SUBSECÇÃO I",
    "Artigo 72.º",
    "Artigo 72.º-A"
]

for value in tests:

    article_match = ARTICLE_RE.match(value)

    if article_match:

        print(
            value,
            "→ ARTICLE",
            article_match.groupdict()
        )

    else:

        print(
            value,
            "→",
            identify_structure(value)
        )

PARTE I → {'type': 'part', 'value': 'PARTE I'}
LIVRO II → {'type': 'book', 'value': 'LIVRO II'}
TÍTULO III → {'type': 'title', 'value': 'TÍTULO III'}
CAPÍTULO IV → {'type': 'chapter', 'value': 'CAPÍTULO IV'}
SECÇÃO II → {'type': 'section', 'value': 'SECÇÃO II'}
SUBSECÇÃO I → {'type': 'subsection', 'value': 'SUBSECÇÃO I'}
Artigo 72.º → ARTICLE {'number': '72.º', 'title': None}
Artigo 72.º-A → ARTICLE {'number': '72.º-A', 'title': None}


## 8. Detecting Table-of-Contents Lines

Tables of contents can contain references that resemble actual article headings and may therefore generate false article detections.

`is_toc_line()` identifies typical index entries, particularly lines containing long dot leaders followed by page references.

These lines can then be excluded from article parsing.

In [13]:
def is_toc_line(line: str) -> bool:
    """
    Identifica linhas típicas de índice/sumário,
    normalmente caracterizadas por vários pontos
    seguidos, eventualmente, do número da página.
    """

    return bool(
        re.search(
            r"\.{4,}\s*\d*\s*$",
            line
        )
    )


def is_toc_page(
    lines: list[str],
    min_toc_lines: int = 5
) -> bool:
    """
    Considera uma página como índice/sumário
    quando contém várias linhas com pontilhados.
    """

    toc_lines = sum(
        1
        for line in lines
        if is_toc_line(line)
    )

    return toc_lines >= min_toc_lines


### 8.1 Testing Table-of-Contents Detection

Representative examples are evaluated to verify that index-like lines are distinguished from genuine article headings and ordinary legal text.

In [14]:
tests_toc = [
    "GERAL ........................................... 25",
    "Artigo 15.º-A",
    "Taxas especiais",
    "OPERACIONAL DA AVALIAÇÃO GERAL ................. 34",
    "CAPÍTULO II"
]

for line in tests_toc:
    print(
        f"{line[:60]:60} "
        f"| TOC: {is_toc_line(line)}"
    )


GERAL ........................................... 25         | TOC: True
Artigo 15.º-A                                                | TOC: False
Taxas especiais                                              | TOC: False
OPERACIONAL DA AVALIAÇÃO GERAL ................. 34          | TOC: True
CAPÍTULO II                                                  | TOC: False


## 9. Parsing a Legal Document

`parse_document()` is the core function of this notebook.

It processes the cleaned text sequentially while maintaining the current legal hierarchy. When a new article heading is detected, the previous article is closed and a new structured article record is created.

For each article, the parser preserves information such as:

- article identifier;
- normalised article number;
- article title;
- starting and ending pages;
- current document hierarchy;
- article text.

When the article title is not present on the same line as the article heading, the parser attempts to infer it from the following suitable line.

The resulting structure provides the basis for article-aware chunking.

In [15]:
def parse_document(document: dict) -> dict:

    document_id = document["document_id"]

    repeated_lines = detect_repeated_page_lines(
        document["pages"]
    )

    hierarchy = {
        "part": None,
        "book": None,
        "title": None,
        "chapter": None,
        "section": None,
        "subsection": None
    }

    hierarchy_order = [
        "part",
        "book",
        "title",
        "chapter",
        "section",
        "subsection"
    ]

    articles = []
    current_article = None

    for page in document["pages"]:

        page_number = page["page"]

        lines = clean_page_lines(
            page["text"],
            repeated_lines
        )

        # Ignorar páginas que parecem corresponder
        # a índices ou sumários
        if is_toc_page(lines):
            continue

        i = 0

        while i < len(lines):

            line = lines[i]

            # ------------------------------------
            # Ignorar linhas de índice/sumário
            # ------------------------------------

            if is_toc_line(line):
                i += 1
                continue

            # ------------------------------------
            # Hierarquia
            # ------------------------------------

            structure = identify_structure(line)

            if structure:

                hierarchy[
                    structure["type"]
                ] = structure["value"]

                index = hierarchy_order.index(
                    structure["type"]
                )

                # limpar níveis inferiores
                for lower_level in hierarchy_order[index + 1:]:
                    hierarchy[lower_level] = None

                i += 1
                continue

            # ------------------------------------
            # Artigo
            # ------------------------------------

            article_match = ARTICLE_RE.match(line)

            if article_match:

                # fechar artigo anterior
                if current_article is not None:

                    current_article["text"] = "\n".join(
                        current_article["text_lines"]
                    ).strip()

                    del current_article["text_lines"]

                    articles.append(current_article)

                article_number = normalize_article_number(
                    article_match.group("number")
                )

                inline_title = article_match.group(
                    "title"
                )

                current_article = {
                    "article": f"Artigo {article_number}",
                    "article_number": article_number,
                    "article_title": inline_title,
                    "page_start": page_number,
                    "page_end": page_number,
                    "hierarchy": hierarchy.copy(),
                    "text_lines": []
                }

                # Se não existir título na mesma linha,
                # tentamos utilizar a linha seguinte
                if (
                    inline_title is None
                    and i + 1 < len(lines)
                ):

                    next_line = lines[i + 1]

                    if (
                        not ARTICLE_RE.match(next_line)
                        and not identify_structure(next_line)
                        and not is_toc_line(next_line)
                        and not re.match(
                            r"^\d+\s*[-.)]",
                            next_line
                        )
                    ):

                        current_article[
                            "article_title"
                        ] = next_line

                        i += 1

                i += 1
                continue

            # ------------------------------------
            # Conteúdo do artigo
            # ------------------------------------

            if current_article is not None:

                current_article[
                    "text_lines"
                ].append(line)

                current_article[
                    "page_end"
                ] = page_number

            i += 1

    # ------------------------------------
    # Fechar último artigo
    # ------------------------------------

    if current_article is not None:

        current_article["text"] = "\n".join(
            current_article["text_lines"]
        ).strip()

        del current_article["text_lines"]

        articles.append(current_article)

    return {
        "document_id": document_id,
        "source_file": document["source_file"],
        "num_pages": document["num_pages"],
        "num_articles": len(articles),
        "articles": articles
    }

## 10. Parser Validation on a Reference Document

Before parsing every document in the corpus, the parser is executed on a reference legal document.

This intermediate validation makes it possible to inspect the number of detected articles and identify obvious parsing problems before batch processing.

In [16]:
test_path = (
    PROCESSED_DIR
    / "CIRS"
    / "extracted.json"
)

with open(test_path, "r", encoding="utf-8") as f:
    document = json.load(f)

parsed = parse_document(document)

print(f"Documento: {parsed['document_id']}")
print(f"Páginas: {parsed['num_pages']}")
print(f"Artigos encontrados: {parsed['num_articles']}")

Documento: CIRS
Páginas: 139
Artigos encontrados: 216


### 10.1 Inspecting Parsed Articles

A sample of parsed articles is displayed with their titles, page ranges, hierarchy, and text.

Manual inspection is useful because legal PDFs can contain formatting variations that are difficult to detect using aggregate metrics alone.

In [17]:
for article in parsed["articles"][:10]:

    print("=" * 80)

    print(article["article"])
    print("Título:", article["article_title"])

    print(
        "Páginas:",
        article["page_start"],
        "-",
        article["page_end"]
    )

    print("Hierarquia:")
    print(article["hierarchy"])

    print()

    print(article["text"][:500])

    print()

Artigo 1.º
Título: Aprovação do Código
Páginas: 7 - 7
Hierarquia:
{'part': None, 'book': None, 'title': None, 'chapter': None, 'section': None, 'subsection': None}

É aprovado o Código do Imposto sobre o Rendimento das Pessoas Singulares (IRS), que faz parte
integrante deste decreto-lei.

Artigo 2.º
Título: Entrada em vigor
Páginas: 7 - 7
Hierarquia:
{'part': None, 'book': None, 'title': None, 'chapter': None, 'section': None, 'subsection': None}

O Código do IRS entra em vigor em 1 de janeiro de 1989.

Artigo 3.º
Título: Impostos abolidos
Páginas: 7 - 7
Hierarquia:
{'part': None, 'book': None, 'title': None, 'chapter': None, 'section': None, 'subsection': None}

1 - Na data da entrada em vigor do Código são abolidos, relativamente aos sujeitos passivos deste
imposto, o imposto profissional, o imposto de capitais, a contribuição industrial, a contribuição predial, o
imposto sobre a indústria agrícola, o imposto complementar, o imposto de mais-valias e o imposto de selo
constante da ver

## 11. Building an Article Validation Table

The parsed articles are converted into a Pandas DataFrame to support systematic validation.

The table includes key metadata and text length, making it easier to identify suspicious records and inspect the overall quality of the parsing stage.

In [18]:
rows = []

for article in parsed["articles"]:

    rows.append({
        "article": article["article"],
        "article_title": article["article_title"],
        "page_start": article["page_start"],
        "page_end": article["page_end"],
        "part": article["hierarchy"]["part"],
        "book": article["hierarchy"]["book"],
        "title": article["hierarchy"]["title"],
        "chapter": article["hierarchy"]["chapter"],
        "section": article["hierarchy"]["section"],
        "subsection": article["hierarchy"]["subsection"],
        "text_length": len(article["text"])
    })

df_articles = pd.DataFrame(rows)

df_articles.head(20)

,article,article_title,page_start,page_end,part,book,title,chapter,section,subsection,text_length
0,Artigo 1.º,Aprovação do Código,7,7,NaN,None,NaN,NaN,NaN,None,123
1,Artigo 2.º,Entrada em vigor,7,7,NaN,None,NaN,NaN,NaN,None,55
2,Artigo 3.º,Impostos abolidos,7,7,NaN,None,NaN,NaN,NaN,None,613
3,Artigo 3.º-A,Regime transitório de enquadramento dos agente...,7,8,NaN,None,NaN,NaN,NaN,None,2316
4,Artigo 4.º,Regime transitório da categoria B,8,8,NaN,None,NaN,NaN,NaN,None,1281
5,Artigo 5.º,Regime transitório da categoria G,8,9,NaN,None,NaN,NaN,NaN,None,1274
6,Artigo 6.º,Reporte de rendimentos,9,9,NaN,None,NaN,NaN,NaN,None,149
7,Artigo 7.º,Obrigação de contabilidade organizada,9,9,NaN,None,NaN,NaN,NaN,None,413
8,Artigo 8.º,Modelos de impressos,9,9,NaN,None,NaN,NaN,NaN,None,143
9,Artigo 9.º,Recibos e Livros,9,9,NaN,None,NaN,NaN,NaN,None,394


### 11.1 Detecting Articles without Text

Articles with zero-length text are identified explicitly.

An empty article may indicate a parsing boundary problem, an unusual source-document structure, or content that was not extracted correctly from the PDF.

In [19]:
df_articles[
    df_articles["text_length"] == 0
]

,article,article_title,page_start,page_end,part,book,title,chapter,section,subsection,text_length


### 11.2 Detecting Articles without Titles

This validation identifies articles for which no title was detected.

A missing title is not necessarily an error, but reviewing these cases helps assess whether title inference is behaving as expected.

In [20]:
df_articles[
    df_articles["article_title"].isna()
][
    [
        "article",
        "page_start",
        "text_length"
    ]
]

,article,page_start,text_length
44,Artigo 103.º,54,50
91,Artigo 128.º,79,389
150,Artigo 11.º,115,1446


### 11.3 Detecting Very Short Articles

Very short article texts are inspected because they may indicate incorrect article boundaries or parsing artefacts.

Some genuine legal provisions are naturally short, so these records are flagged for review rather than automatically discarded.

In [21]:
df_articles[
    df_articles["text_length"] < 50
][
    [
        "article",
        "article_title",
        "page_start",
        "text_length"
    ]
]

,article,article_title,page_start,text_length
86,Artigo 56.º,Abatimentos ao rendimento líquido total,75,14
123,Artigo 79.º,"Deduções dos sujeitos passivos, descendentes e...",105,11
124,Artigo 80.º,Crédito de imposto por dupla tributação económica,105,11
126,Artigo 82.º,Despesas de saúde,107,11
127,Artigo 83.º,Despesas de educação e formação,107,11
130,Artigo 85.º,Encargos com imóveis,108,11
131,Artigo 85.º-A,Deduções ambientais,108,11
132,Artigo 86.º,Prémios de seguros,108,15
134,Artigo 88.º,Benefícios fiscais,109,15
169,Artigo 109.º,Compensação,125,11


### 11.4 Inspecting the Longest Articles

The longest parsed articles are displayed to identify unusually large records.

Very large article texts can reveal missed article boundaries and are also relevant for the chunking strategy implemented in the next notebook.

In [22]:
df_articles.sort_values(
    "text_length",
    ascending=False
)[
    [
        "article",
        "article_title",
        "page_start",
        "page_end",
        "text_length"
    ]
].head(20)

,article,article_title,page_start,page_end,text_length
16,Artigo 16.º,As modificações do Código,11,23,44331
28,Artigo 10.º,Mais-valias,34,40,23012
107,Artigo 72.º,Taxas especiais,86,91,19340
18,Artigo 2.º,Rendimentos da categoria A,23,26,11578
54,Artigo 31.º,Regime simplificado,61,64,10932
45,Artigo 22.º,Englobamento,54,56,10771
125,Artigo 81.º,Eliminação da dupla tributação jurídica intern...,105,107,9880
37,Artigo 16.º,Residência,48,50,8237
154,Artigo 101.º,Retenção sobre rendimentos de outras categorias,117,119,7940
179,Artigo 119.º,Comunicação de rendimentos e retenções,129,131,7510


## 12. Saving Parsed Documents

`save_parsed_document()` stores the structured representation of each legal document as:

```text
data/processed/<DOCUMENT_ID>/parsed.json
```

Keeping parsing output separate from extraction output makes the ingestion pipeline modular and allows individual stages to be rerun independently.

In [23]:
def save_parsed_document(
    parsed_document: dict,
    output_dir: Path
) -> Path:

    document_id = parsed_document["document_id"]

    document_dir = output_dir / document_id

    document_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    output_path = (
        document_dir
        / "parsed.json"
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            parsed_document,
            f,
            ensure_ascii=False,
            indent=2
        )

    return output_path

## 13. Parsing the Complete Corpus

After the parser has been validated, every discovered `extracted.json` file is processed.

Each document is handled independently. The resulting parsed representation is saved to its corresponding document directory, and the processing outcome is recorded for later inspection.

In [24]:
results = []

for extracted_path in extracted_files:

    document_id = extracted_path.parent.name

    print(
        f"A processar {document_id}...",
        end=" "
    )

    try:

        with open(
            extracted_path,
            "r",
            encoding="utf-8"
        ) as f:

            document = json.load(f)

        parsed_document = parse_document(
            document
        )

        output_path = save_parsed_document(
            parsed_document,
            PROCESSED_DIR
        )

        results.append({
            "document_id": document_id,
            "pages": parsed_document[
                "num_pages"
            ],
            "articles": parsed_document[
                "num_articles"
            ],
            "status": "OK"
        })

        print(
            f"OK "
            f"({parsed_document['num_articles']} artigos)"
        )

    except Exception as e:

        results.append({
            "document_id": document_id,
            "pages": None,
            "articles": None,
            "status": f"ERROR: {e}"
        })

        print(f"ERRO: {e}")

A processar CIMI... OK (192 artigos)
A processar CIRC... OK (171 artigos)
A processar CIRS... OK (216 artigos)
A processar CIS... OK (93 artigos)
A processar CIVA... OK (218 artigos)
A processar EBF... OK (183 artigos)
A processar LGT... OK (124 artigos)
A processar OE2026... OK (277 artigos)
A processar RGIT... OK (147 artigos)
A processar RITI... OK (151 artigos)


### 13.1 Corpus Processing Results

The batch-processing results are converted into a DataFrame.

This provides a compact overview of the documents processed successfully and the number of articles detected in each legal source.

In [25]:
df_results = pd.DataFrame(results)

df_results

,document_id,pages,articles,status
0,CIMI,91,192,OK
1,CIRC,164,171,OK
2,CIRS,139,216,OK
3,CIS,64,93,OK
4,CIVA,188,218,OK
5,EBF,107,183,OK
6,LGT,65,124,OK
7,OE2026,270,277,OK
8,RGIT,73,147,OK
9,RITI,74,151,OK


### 13.2 Total Article Count

The total number of identified articles across the complete corpus is calculated.

This provides a useful corpus-level reference that can later be compared with the number of generated chunks and indexed vectors.

In [26]:
total_articles = df_results[
    "articles"
].fillna(0).sum()

print(
    f"Total de artigos identificados: "
    f"{int(total_articles)}"
)

Total de artigos identificados: 1772


## 14. Duplicate Article Validation

Article identifiers should be unique within each legal document.

This validation searches the generated `parsed.json` files for duplicate article identifiers. Duplicate records can indicate false detections, table-of-contents contamination, or inconsistencies in article-number normalisation.

Identifying duplicates at this stage is important because article identifiers contribute to chunk metadata and chunk ID generation.

In [27]:
duplicate_summary = []

for parsed_path in sorted(
    PROCESSED_DIR.glob("*/parsed.json")
):

    with open(
        parsed_path,
        "r",
        encoding="utf-8"
    ) as f:
        data = json.load(f)

    article_numbers = [
        article["article_number"]
        for article in data["articles"]
    ]

    counts = Counter(article_numbers)

    duplicates = {
        article: count
        for article, count in counts.items()
        if count > 1
    }

    duplicate_summary.append({
        "document_id": data["document_id"],
        "articles": len(article_numbers),
        "duplicate_articles": len(duplicates)
    })


df_duplicates_summary = pd.DataFrame(
    duplicate_summary
)

df_duplicates_summary


,document_id,articles,duplicate_articles
0,CIMI,192,18
1,CIRC,171,2
2,CIRS,216,18
3,CIS,93,20
4,CIVA,218,71
5,EBF,183,35
6,LGT,124,6
7,OE2026,277,13
8,RGIT,147,13
9,RITI,151,34


## 15. Final Parsed-Corpus Validation

The generated parsed documents are reopened for a final consistency inspection.

This step validates the persisted output rather than only the in-memory structures created during parsing, helping ensure that the files written to disk contain the expected article representation.

In [28]:
for parsed_path in sorted(
    PROCESSED_DIR.glob("*/parsed.json")
):

    with open(
        parsed_path,
        "r",
        encoding="utf-8"
    ) as f:
        data = json.load(f)

    counts = Counter(
        article["article_number"]
        for article in data["articles"]
    )

    duplicates = {
        article: count
        for article, count in counts.items()
        if count > 1
    }

    if duplicates:

        print(
            f"\n{data['document_id']}"
        )

        for article, count in duplicates.items():
            print(
                f"  {article}: "
                f"{count} ocorrências"
            )



CIMI
  13.º: 2 ocorrências
  14.º: 2 ocorrências
  15.º: 2 ocorrências
  16.º: 2 ocorrências
  18.º: 2 ocorrências
  19.º: 2 ocorrências
  20.º: 2 ocorrências
  21.º: 2 ocorrências
  22.º: 2 ocorrências
  23.º: 2 ocorrências
  24.º: 2 ocorrências
  25.º: 3 ocorrências
  26.º: 2 ocorrências
  27.º: 2 ocorrências
  28.º: 2 ocorrências
  29.º: 2 ocorrências
  30.º: 2 ocorrências
  32.º: 2 ocorrências

CIRC
  73.º: 2 ocorrências
  91.º: 2 ocorrências

CIRS
  1.º: 2 ocorrências
  2.º: 2 ocorrências
  3.º: 2 ocorrências
  4.º: 2 ocorrências
  5.º: 2 ocorrências
  6.º: 2 ocorrências
  7.º: 2 ocorrências
  8.º: 2 ocorrências
  9.º: 2 ocorrências
  10.º: 2 ocorrências
  11.º: 3 ocorrências
  12.º: 2 ocorrências
  13.º: 2 ocorrências
  14.º: 2 ocorrências
  15.º: 2 ocorrências
  16.º: 2 ocorrências
  103.º: 2 ocorrências
  128.º: 2 ocorrências

CIS
  13.º: 2 ocorrências
  14.º: 2 ocorrências
  15.º: 2 ocorrências
  16.º: 2 ocorrências
  17.º: 2 ocorrências
  18.º: 2 ocorrências
  19.º: 2 ocorrê

### 15.1 Targeted Inspection of CIMI

A targeted inspection of the parsed CIMI document is performed as an additional validation step.

This is useful for examining known structural edge cases, particularly amended or inserted article numbers, and confirming that the parser has preserved them correctly.

## Next Step

The structured article files generated by this notebook are the input to:

```text
03_document_chunking.ipynb
```

The next stage transforms the parsed legal articles into retrieval-oriented chunks while preserving document, article, hierarchy, and page metadata.

In [29]:
cimi_path = (
    PROCESSED_DIR
    / "CIMI"
    / "parsed.json"
)

if cimi_path.exists():

    with open(
        cimi_path,
        "r",
        encoding="utf-8"
    ) as f:
        cimi = json.load(f)

    for article in cimi["articles"][:50]:

        print(
            f"{article['article']:15} "
            f"| {str(article['article_title'])[:60]:60} "
            f"| página {article['page_start']}"
        )

else:
    print("CIMI/parsed.json não encontrado.")


Artigo 13.º     | Elementos para avaliações                                    | página 8
Artigo 14.º     | Taxas de conservação de esgotos                              | página 8
Artigo 15.º     | Avaliação de prédios já inscritos na matriz                  | página 8
Artigo 15.º-A   | (Aditado pela Lei nº 60-A/2011, de 30 de novembro)           | página 9
Artigo 15.º-B   | (Aditado pela Lei nº 60-A/2011, de 30 de novembro)           | página 9
Artigo 15.º-C   | (Aditado pela Lei nº 60-A/2011, de 30 de novembro)           | página 10
Artigo 15.º-D   | (Aditado pela Lei nº 60-A/2011, de 30 de novembro)           | página 10
Artigo 15.º-E   | (Aditado pela Lei nº 60-A/2011, de 30 de novembro)           | página 11
Artigo 15.º-F   | (Aditado pela Lei nº 60-A/2011, de 30 de novembro)           | página 11
Artigo 15.º-G   | (Aditado pela Lei nº 60-A/2011, de 30 de novembro)           | página 11
Artigo 15.º-H   | (Aditado pela Lei nº 60-A/2011, de 30 de novembro)           | página 12
Arti